# Numerai Memory Efficient Pipeline

**Automated Numerai submissions with feature engineering**

## Configuration

In [1]:
from numerapi import NumerAPI
from pathlib import Path
import os

PUBLIC_ID = 'KY2YNIU7TAALGQRVVTHERFRG7QUJP3FJ'
SECRET_KEY = 'EIMJDP62GSPHETVXXZGLFLOOGKFBRBA6DYPJ7WWSVRMUR3OFKGPIOVVZYHWXBHP4'

napi = NumerAPI(PUBLIC_ID, SECRET_KEY)

# Delete old live data
live_file = "v5.2/live.parquet"
if Path(live_file).exists():
    os.remove(live_file)
    print("Deleted old live.parquet")

# Download fresh data for current round
print(f"Downloading fresh live data for round {napi.get_current_round()}...")
napi.download_dataset(live_file)
print("✅ Fresh live data downloaded!")

Deleted old live.parquet


2026-01-07 13:44:21,985 INFO numerapi.utils: starting download
v5.2/live.parquet: 9.47MB [00:04, 2.16MB/s]                                                                            

✅ Fresh live data downloaded!


In [2]:
# Pipeline Settings
TRAIN_SAMPLE_SIZE = 200000
NUM_FEATURES = 2000
USE_FEATURE_ENGINEERING = True
USE_NEURAL_NETWORK = True

# File paths
TRAIN_FILE = 'v5.2/train.parquet'
VAL_FILE = 'v5.2/validation.parquet'
LIVE_FILE = 'v5.2/live.parquet'

print('✓ Configuration loaded')

✓ Configuration loaded


In [3]:
# Auto-install PyTorch if missing
try:
    import torch
    print(f'✓ PyTorch {torch.__version__} already installed')
except ImportError:
    print('⚠️  PyTorch not found. Installing...')
    import subprocess
    import sys
    
    try:
        # Install CPU-only version (smaller, faster)
        print('Installing PyTorch (CPU version)...')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', 
            'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'
        ])
        
        # Try importing again
        import torch
        print(f'✓ PyTorch {torch.__version__} installed successfully!')
        
        # Update global variables
        import torch.nn as nn
        import torch.optim as optim
        from torch.utils.data import Dataset, DataLoader
        PYTORCH_AVAILABLE = True
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'✓ Device: {device}')
        
    except Exception as e:
        print(f'✗ PyTorch installation failed: {e}')
        print('Continuing without Neural Network...')
        PYTORCH_AVAILABLE = False
        USE_NEURAL_NETWORK = False
        device = None

✓ PyTorch 2.9.1+cpu already installed


## API Configuration

In [4]:
# ADD YOUR API KEYS
NUMERAI_PUBLIC_KEY = 'KY2YNIU7TAALGQRVVTHERFRG7QUJP3FJ'  # Get from https://numer.ai/settings
NUMERAI_SECRET_KEY = 'EIMJDP62GSPHETVXXZGLFLOOGKFBRBA6DYPJ7WWSVRMUR3OFKGPIOVVZYHWXBHP4'
# Model mappings (pre-configured)
MODEL_UPLOADS = {
    'submission_enhanced_ensemble_rank': {'name': 'jewellzilla_rank_ens', 'model_id': 'ed68de39-57f8-43d2-92ce-3a6e375f4b8a'},
    'submission_enhanced_lgbm': {'name': 'jewellzilla_std_enh', 'model_id': 'e02eda5d-1760-4984-a6c2-526a876621fb'},
    'submission_enhanced_xgboost': {'name': 'jewellzilla_xg_enh', 'model_id': '98639631-99f8-4139-b069-21fdc094a074'},
    'submission_enhanced_neural_net': {'name': 'jewellzilla_nn', 'model_id': '548df016-74d2-4e9b-9164-ee4165e19a7e'}
}
AUTO_UPLOAD = True  # Set to True to enable
print('✓ API configuration loaded')




✓ API configuration loaded


## Imports

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import pickle
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from numerai_feature_engineering import NumeraiFeatureEngineer, get_all_feature_columns

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    PYTORCH_AVAILABLE = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'✓ PyTorch available: {device}')
except ImportError:
    PYTORCH_AVAILABLE = False
    USE_NEURAL_NETWORK = False
    device = None
    print('⚠️  PyTorch not available')

print('✓ Imports complete')

✓ PyTorch available: cpu
✓ Imports complete


## Define Model Classes

In [6]:
if PYTORCH_AVAILABLE:
    class NumeraiDataset(Dataset):
        def __init__(self, features, targets):
            self.features = torch.FloatTensor(features)
            self.targets = torch.FloatTensor(targets)
        
        def __len__(self):
            return len(self.features)
        
        def __getitem__(self, idx):
            return self.features[idx], self.targets[idx]

    class NumeraiNN(nn.Module):
        def __init__(self, input_dim):
            super(NumeraiNN, self).__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(512, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, 1),
                nn.Sigmoid()
            )
        
        def forward(self, x):
            return self.network(x)
    
    print('✓ PyTorch classes defined')
else:
    print('✓ Skipping PyTorch classes')

✓ PyTorch classes defined


## Load Data

In [7]:
print('='*70)
print('LOADING DATA')
print('='*70)

print(f'\nLoading {TRAIN_FILE}...')
full_train = pd.read_parquet(TRAIN_FILE)
print(f'  Full size: {len(full_train):,} rows')

if len(full_train) > TRAIN_SAMPLE_SIZE:
    train_data = full_train.sample(n=TRAIN_SAMPLE_SIZE, random_state=42)
    print(f'  Sampled to: {len(train_data):,} rows')
    del full_train
else:
    train_data = full_train

print(f'\nLoading {VAL_FILE}...')
full_val = pd.read_parquet(VAL_FILE)
print(f'  Full size: {len(full_val):,} rows')

if len(full_val) > TRAIN_SAMPLE_SIZE:
    val_data = full_val.sample(n=TRAIN_SAMPLE_SIZE, random_state=42)
    print(f'  Sampled to: {len(val_data):,} rows')
    del full_val
else:
    val_data = full_val

print(f'\nLoading {LIVE_FILE}...')
live_data = pd.read_parquet(LIVE_FILE)

print(f'\n✓ Training: {train_data.shape}')
print(f'✓ Validation: {val_data.shape}')
print(f'✓ Live: {live_data.shape}')

LOADING DATA

Loading v5.2/train.parquet...
  Full size: 2,746,268 rows
  Sampled to: 200,000 rows

Loading v5.2/validation.parquet...
  Full size: 3,882,191 rows
  Sampled to: 200,000 rows

Loading v5.2/live.parquet...

✓ Training: (200000, 2791)
✓ Validation: (200000, 2791)
✓ Live: (6651, 2791)


## Feature Engineering

In [8]:
print('\n' + '='*70)
print('FEATURE ENGINEERING')
print('='*70)

if USE_FEATURE_ENGINEERING:
    if os.path.exists('feature_engineer.pkl'):
        print('Loading existing feature engineer...')
        feature_engineer = NumeraiFeatureEngineer.load('feature_engineer.pkl')
        train_data = feature_engineer.transform(train_data)
        val_data = feature_engineer.transform(val_data)
        live_data = feature_engineer.transform(live_data)
    else:
        print('Creating new feature engineer...')
        feature_engineer = NumeraiFeatureEngineer()
        train_data = feature_engineer.fit_transform(train_data)
        val_data = feature_engineer.transform(val_data)
        live_data = feature_engineer.transform(live_data)
        feature_engineer.save('feature_engineer.pkl')
    
    all_feature_cols = get_all_feature_columns(train_data)
    print(f'Total features: {len(all_feature_cols)}')
    print(f'Selecting top {NUM_FEATURES}...')
    
    correlations = train_data[all_feature_cols].corrwith(train_data['target']).abs()
    feature_cols = correlations.nlargest(NUM_FEATURES).index.tolist()
    print(f'✓ Using {len(feature_cols)} features')
else:
    all_features = [f for f in train_data.columns if f.startswith('feature_')]
    correlations = train_data[all_features].corrwith(train_data['target']).abs()
    feature_cols = correlations.nlargest(NUM_FEATURES).index.tolist()
    print(f'✓ Using {len(feature_cols)} original features')


FEATURE ENGINEERING
Loading existing feature engineer...
✓ Feature engineer loaded from feature_engineer.pkl

TRANSFORMING DATA WITH FITTED PIPELINE
  - Creating charisma interactions...
  - Creating cross-group features...
  - Creating rank features...
  - Creating polynomial features...
  - Creating era-aware features...
  - Creating binned features...
  - Creating group aggregates...
  - Creating PCA features...
✓ Transformation complete


TRANSFORMING DATA WITH FITTED PIPELINE
  - Creating charisma interactions...
  - Creating cross-group features...
  - Creating rank features...
  - Creating polynomial features...
  - Creating era-aware features...
  - Creating binned features...
  - Creating group aggregates...
  - Creating PCA features...
✓ Transformation complete


TRANSFORMING DATA WITH FITTED PIPELINE
  - Creating charisma interactions...
  - Creating cross-group features...
  - Creating rank features...
  - Creating polynomial features...
  - Creating era-aware features...


## Train LightGBM

In [9]:
print('\n' + '='*70)
print('TRAINING LIGHTGBM')
print('='*70)

lgbm_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=32,
    colsample_bytree=0.1,
    random_state=42,
    n_jobs=-1
)

X_train = train_data[feature_cols].values
y_train = train_data['target'].values
X_val = val_data[feature_cols].values
y_val = val_data['target'].values

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

lgbm_model.booster_.save_model('jewellzilla_lgbm_enhanced.txt')
print('\n✓ LightGBM saved')


TRAINING LIGHTGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.118036 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10004
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 2000
[LightGBM] [Info] Start training from score 0.499698
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.05185
Early stopping, best iteration is:
[89]	valid_0's l2: 0.0518496

✓ LightGBM saved


## Train XGBoost

In [10]:
print('\n' + '='*70)
print('TRAINING XGBOOST')
print('='*70)
try:
    xgb_model = xgb.XGBRegressor(
        n_estimators=2000,
        learning_rate=0.01,
        max_depth=5,
        colsample_bytree=0.1,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50
    )
    # Remove NaN
    train_mask = ~(np.isnan(X_train).any(axis=1) | np.isnan(y_train))
    val_mask = ~(np.isnan(X_val).any(axis=1) | np.isnan(y_val))
    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    X_val_clean = X_val[val_mask]
    y_val_clean = y_val[val_mask]
    print(f'Cleaned: {len(X_train_clean)} train, {len(X_val_clean)} val')
    xgb_model.fit(
        X_train_clean, y_train_clean,
        eval_set=[(X_val_clean, y_val_clean)],
        verbose=100
    )
    xgb_model.save_model('jewellzilla_xgb_enhanced.json')
    print('\n✓ XGBoost saved')
    xgb_success = True
except Exception as e:
    print(f'⚠️  XGBoost failed: {e}')
    xgb_model = None
    xgb_success = False



TRAINING XGBOOST
Cleaned: 200000 train, 197989 val
[0]	validation_0-rmse:0.22326
[100]	validation_0-rmse:0.22324
[150]	validation_0-rmse:0.22324

✓ XGBoost saved


## Train Neural Network

In [11]:
if USE_NEURAL_NETWORK and PYTORCH_AVAILABLE:
    print('\n' + '='*70)
    print('TRAINING NEURAL NETWORK')
    print('='*70)
    
    try:
        train_mask = ~(np.isnan(X_train).any(axis=1) | np.isnan(y_train))
        val_mask = ~(np.isnan(X_val).any(axis=1) | np.isnan(y_val))
        
        X_train_clean = X_train[train_mask]
        y_train_clean = y_train[train_mask]
        X_val_clean = X_val[val_mask]
        y_val_clean = y_val[val_mask]
        
        train_dataset = NumeraiDataset(X_train_clean, y_train_clean)
        val_dataset = NumeraiDataset(X_val_clean, y_val_clean)
        
        train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=2048)
        
        nn_model = NumeraiNN(len(feature_cols)).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(nn_model.parameters(), lr=0.001)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)
        
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(100):
            nn_model.train()
            train_loss = 0
            for features, targets in train_loader:
                features, targets = features.to(device), targets.to(device)
                optimizer.zero_grad()
                outputs = nn_model(features)
                loss = criterion(outputs.squeeze(), targets)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            nn_model.eval()
            val_loss = 0
            with torch.no_grad():
                for features, targets in val_loader:
                    features, targets = features.to(device), targets.to(device)
                    outputs = nn_model(features)
                    loss = criterion(outputs.squeeze(), targets)
                    val_loss += loss.item()
            
            avg_val_loss = val_loss / len(val_loader)
            
            if (epoch + 1) % 5 == 0:
                print(f'Epoch {epoch+1}: Val Loss={avg_val_loss:.6f}')
            
            scheduler.step(avg_val_loss)
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                torch.save(nn_model.state_dict(), 'jewellzilla_nn_enhanced.pth')
            else:
                patience_counter += 1
                if patience_counter >= 10:
                    break
        
        print('\n✓ Neural Network saved')
        nn_success = True
    except Exception as e:
        print(f'⚠️  Neural Network failed: {e}')
        nn_model = None
        nn_success = False
else:
    nn_model = None
    nn_success = False


TRAINING NEURAL NETWORK
Epoch 5: Val Loss=0.050150
Epoch 10: Val Loss=0.053264

✓ Neural Network saved


## Generate Predictions

In [12]:
print('\n' + '='*70)
print('GENERATING PREDICTIONS')
print('='*70)

X_live = live_data[feature_cols].values
predictions = pd.DataFrame(index=live_data.index)

print('LightGBM...')
predictions['lgbm'] = lgbm_model.predict(X_live)

if xgb_success:
    print('XGBoost...')
    predictions['xgboost'] = xgb_model.predict(X_live)

if nn_success:
    print('Neural Network...')
    nn_model.eval()
    with torch.no_grad():
        X_live_tensor = torch.FloatTensor(X_live).to(device)
        nn_preds = []
        for i in range(0, len(X_live_tensor), 10000):
            batch = X_live_tensor[i:i+10000]
            nn_preds.extend(nn_model(batch).cpu().numpy().flatten())
        predictions['neural_net'] = nn_preds

print(f'\n✓ Predictions from {len(predictions.columns)} models')


GENERATING PREDICTIONS
LightGBM...
XGBoost...
Neural Network...

✓ Predictions from 3 models


## Create Ensembles

In [13]:
model_cols = predictions.columns.tolist()

predictions['ensemble_mean'] = predictions[model_cols].mean(axis=1)

if len(model_cols) == 3:
    weights = {'lgbm': 0.4, 'xgboost': 0.4, 'neural_net': 0.2}
elif len(model_cols) == 2:
    weights = {'lgbm': 0.5, 'xgboost': 0.5}
else:
    weights = {col: 1.0/len(model_cols) for col in model_cols}

predictions['ensemble_weighted'] = sum(
    predictions[m] * w for m, w in weights.items() if m in predictions.columns
)

for col in model_cols:
    predictions[f'{col}_rank'] = predictions[col].rank(pct=True)

rank_cols = [f'{col}_rank' for col in model_cols]
predictions['ensemble_rank'] = predictions[rank_cols].mean(axis=1)

print('✓ Created ensembles')

✓ Created ensembles


## Save Submissions

In [14]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Create submissions folder if it doesn't exist
import os
os.makedirs('submissions', exist_ok=True)

submission_types = {
    'lgbm': 'lgbm',
    'xgboost': 'xgboost',
    'neural_net': 'neural_net',
    'ensemble_mean': 'ensemble_mean',
    'ensemble_weighted': 'ensemble_weighted',
    'ensemble_rank': 'ensemble_rank'
}

saved_files = []

for pred_col, name in submission_types.items():
    if pred_col in predictions.columns:
        filename = f'submissions/submission_enhanced_{name}_{timestamp}.csv'
        submission = predictions[[pred_col]].rename(columns={pred_col: 'prediction'})
        submission.to_csv(filename)
        saved_files.append(filename)
        print(f'  ✓ {filename}')

print(f'\n✓ Saved {len(saved_files)} files to submissions/ folder')



  ✓ submissions/submission_enhanced_lgbm_20260107_135838.csv
  ✓ submissions/submission_enhanced_xgboost_20260107_135838.csv
  ✓ submissions/submission_enhanced_neural_net_20260107_135838.csv
  ✓ submissions/submission_enhanced_ensemble_mean_20260107_135838.csv
  ✓ submissions/submission_enhanced_ensemble_weighted_20260107_135838.csv
  ✓ submissions/submission_enhanced_ensemble_rank_20260107_135838.csv

✓ Saved 6 files to submissions/ folder


## Upload to Numerai

In [15]:
if AUTO_UPLOAD and NUMERAI_PUBLIC_KEY and NUMERAI_SECRET_KEY:
    print('\n' + '='*70)
    print('UPLOADING TO NUMERAI')
    print('='*70)
    
    try:
        from numerapi import NumerAPI
        napi = NumerAPI(NUMERAI_PUBLIC_KEY, NUMERAI_SECRET_KEY)
        print('✓ Connected\n')
        
        for file_prefix, model_info in MODEL_UPLOADS.items():
            matching = [f for f in saved_files if file_prefix in f]
            if matching:
                try:
                    print(f"Uploading {model_info['name']}...")
                    submission_id = napi.upload_predictions(matching[0], model_id=model_info['model_id'])
                    print(f"  ✓ Success! ID: {submission_id}\n")
                except Exception as e:
                    print(f"  ✗ Failed: {e}\n")
        
        print('✓ Upload complete!')
        print('Check: https://numer.ai/models')
    except Exception as e:
        print(f'Upload error: {e}')
else:
    print('\n⚠️  Auto-upload disabled')
    print('Enable by setting AUTO_UPLOAD=True and adding API keys')


2026-01-07 13:58:38,793 INFO numerapi.base_api: uploading predictions...



UPLOADING TO NUMERAI
✓ Connected

Uploading jewellzilla_rank_ens...


2026-01-07 13:58:45,021 INFO numerapi.base_api: uploading predictions...


  ✓ Success! ID: 92c4931e-9142-47a7-9066-bc2830b6e553

Uploading jewellzilla_std_enh...


2026-01-07 13:58:51,670 INFO numerapi.base_api: uploading predictions...


  ✓ Success! ID: bfc67943-3cea-467f-a844-36f146173952

Uploading jewellzilla_xg_enh...


2026-01-07 13:58:57,532 INFO numerapi.base_api: uploading predictions...


  ✓ Success! ID: 7a0a03e8-f6f7-4a18-bc10-1e3f677b5eb4

Uploading jewellzilla_nn...
  ✓ Success! ID: fad06cc6-d439-43c7-a11f-152f8d6e9254

✓ Upload complete!
Check: https://numer.ai/models


## Summary

In [16]:
print('\n' + '='*70)
print('PIPELINE COMPLETE!')
print('='*70)
print(f'\nModels trained: {len([m for m in [lgbm_model, xgb_success, nn_success] if m])}')
print(f'Features used: {len(feature_cols)}')
print(f'Submissions: {len(saved_files)}')
print(f'\nRecommended: submission_ensemble_rank_{timestamp}.csv')
print('='*70)


PIPELINE COMPLETE!

Models trained: 3
Features used: 2000
Submissions: 6

Recommended: submission_ensemble_rank_20260107_135838.csv


In [17]:
from numerapi import NumerAPI
# Initialize with your credentials
PUBLIC_ID = 'KY2YNIU7TAALGQRVVTHERFRG7QUJP3FJ'
SECRET_KEY = 'EIMJDP62GSPHETVXXZGLFLOOGKFBRBA6DYPJ7WWSVRMUR3OFKGPIOVVZYHWXBHP4'
napi = NumerAPI(PUBLIC_ID, SECRET_KEY)
# Your model IDs (UUIDs)
models = {
    "jewellzilla_rank_ens": "ed68de39-57f8-43d2-92ce-3a6e375f4b8a",
    "jewellzilla_std_enh": "e02eda5d-1760-4984-a6c2-526a876621fb",
    "jewellzilla_xg_enh": "98639631-99f8-4139-b069-21fdc094a074",
    "jewellzilla_nn": "548df016-74d2-4e9b-9164-ee4165e19a7e",
}
# Get current round
current_round = napi.get_current_round()
print(f"Current round: {current_round}")
print("="*70)
# Check each model's latest submission
for model_name, model_id in models.items():
    try:
        print(f"\n{model_name}:")
        # Get submission filenames for this model
        submissions = napi.get_submission_filenames(model_id=model_id)
        if submissions:
            print(f"  Latest submission: {submissions[0]}")
            print(f"  Total submissions: {len(submissions)}")
        else:
            print(f"  No submissions found")
    except Exception as e:
        print(f"  Error: {e}")
# Or get detailed info about your account
print("\n" + "="*70)
print("ACCOUNT SUMMARY")
print("="*70)
try:
    account = napi.get_account()
    models_info = account.get('models', [])
    for model in models_info:
        print(f"\n{model['name']}:")
        print(f"  ID: {model['id']}")
        print(f"  Status: {model.get('status', 'unknown')}")
        # Get latest round performance
        if 'latest_ranks' in model:
            print(f"  Latest performance: {model['latest_ranks']}")
except Exception as e:
    print(f"Error getting account info: {e}")


Current round: 1176

jewellzilla_rank_ens:
  Latest submission: {'round_num': 1172, 'tournament': 8, 'filename': 'submission_ensemble_rank_20260101_215632-01LxieA00ncU.csv'}
  Total submissions: 5

jewellzilla_std_enh:
  Latest submission: {'round_num': 1172, 'tournament': 8, 'filename': 'submission_lgbm_20260101_215632-MEsidWf4rUX1.csv'}
  Total submissions: 5

jewellzilla_xg_enh:
  Latest submission: {'round_num': 1172, 'tournament': 8, 'filename': 'submission_xgboost_20260101_215632-aQqTpLTAFG1G.csv'}
  Total submissions: 5

jewellzilla_nn:
  Latest submission: {'round_num': 1172, 'tournament': 8, 'filename': 'submission_neural_net_20260101_215632-TBzpGKHty6aA.csv'}
  Total submissions: 5

ACCOUNT SUMMARY

jewellzilla_std:
  ID: bd2f8540-d90a-4206-b1c5-4e28f2865cba
  Status: unknown

jz_mar2021:
  ID: 8c90a670-f2cf-49d1-838c-f4b248b3c8ed
  Status: unknown

jewellzilla_cat:
  ID: 9e253cd6-6b6b-4178-a641-c9738f21eb11
  Status: unknown

jewellzilla_xg:
  ID: a65acf61-b5ba-4982-a7c8-733